In [4]:
# Importation des modules
import socket       # Pour la communication réseau (TCP)
import secrets      # Pour la génération sécurisée de nombres aléatoires
import os           # Pour générer la clé AES
import hashlib      # Pour le hachage SHA-256
import time

# Configuration de l'adresse et du port de connexion
hote = '127.0.0.1'  # Adresse du serveur
port = 65432        # Port d'écoute du serveur

# ------------------------
# --- Question 1 : Établissement du secret partagé via Diffie-Hellman ---
# Alice génère une clé privée, calcule sa clé publique, envoie celle-ci à Bob,
# reçoit celle de Bob, puis calcule un secret partagé commun.

# 1. Génération de la paire de clés Diffie-Hellman côté Alice
premier = 0xFFFFFFFB
generateur = 2 #Vu avec le prof
cle_privee_alice = secrets.randbelow(premier)
cle_publique_alice = pow(generateur, cle_privee_alice, premier)

# Connexion au serveur Bob
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as client:
    client.connect((hote, port))

    # Envoi de la clé publique d'Alice
    client.sendall(str(cle_publique_alice).encode())

    # Réception de la clé publique de Bob
    cle_publique_bob = int(client.recv(4096).decode())

    # Calcul du secret partagé
    secret_partage = pow(cle_publique_bob, cle_privee_alice, premier)
    secret_partage_octets = secret_partage.to_bytes(16, byteorder='big')

    # ------------------------
    # --- Question 2 : Génération et chiffrement de la clé AES ---
    # Alice génère une clé AES, puis la chiffre avec le secret partagé (XOR).

    cle_aes = os.urandom(16)
    cle_aes_chiffree = bytes([k ^ s for k, s in zip(cle_aes, secret_partage_octets)])

    # ------------------------
    # --- Question 3 : Génération du hash SHA-256 de la clé AES ---
    # Alice génère le hash de la clé AES en clair pour permettre à Bob de vérifier son intégrité.

    hash_aes = hashlib.sha256(cle_aes).digest()

    # Envoi de la clé AES chiffrée + hash
    client.sendall(cle_aes_chiffree + hash_aes)

    # ------------------------
    # --- Question 4 : Réception et vérification du message de confirmation ---
    # Alice reçoit un message chiffré de Bob et le déchiffre avec 
    # la clé AES pour confirmer que Bob a bien reçu la bonne clé.

    message_chiffre = client.recv(4096)
    message_dechiffre = bytes([c ^ k for c, k in zip(message_chiffre, cle_aes)]).rstrip(b'\x00')

    if message_dechiffre == b"CBON":
        print("\n Alice confirme que Bob a bien la bonne clé AES.")
    else:
        print("\n Erreur : la confirmation n'est pas valide.")



 Alice confirme que Bob a bien la bonne clé AES.
